In [10]:
import pandas as pd

X_train = pd.read_csv("C:/Users/user/Downloads/X_train_update.csv")
y_train = pd.read_csv("C:/Users/user/Downloads/Y_train_CVw08PX.csv")

df = X_train.merge(y_train, left_index=True, right_on="Unnamed: 0")
df.head()

,Unnamed: 0,Unnamed: 0_x,designation,description,productid,imageid,Unnamed: 0_y,prdtypecode
0,0,0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,0,10
1,1,1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,1,2280
2,2,2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,2,50
3,3,3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,3,1280
4,4,4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,4,2705


In [11]:
# Import de la bibliothèque pour les expressions régulières
import re

# Import de BeautifulSoup pour supprimer les balises HTML
from bs4 import BeautifulSoup

# Import de pandas pour manipuler le dataframe
import pandas as pd

# Fonction qui nettoie un texte
def clean_text(text):

    # Si le texte est NaN (valeur manquante dans pandas)
    # on retourne une chaine vide pour éviter les erreurs
    if pd.isna(text):
        return ""

    # Supprime les balises HTML présentes dans les descriptions produits
    # Exemple : <p>Produit</p> -> Produit
    text = BeautifulSoup(text, "html.parser").get_text()

    # Convertit tout le texte en minuscules
    # Exemple : "Robot Piscine" -> "robot piscine"
    text = text.lower()

    # Supprime tous les caractères spéciaux
    # On garde seulement :
    # - lettres
    # - lettres accentuées
    # - chiffres
    # - espaces
    # Tout le reste est remplacé par un espace
    text = re.sub(r"[^a-zA-Zàâäéèêëîïôöùûüç0-9 ]", " ", text)

    # Remplace plusieurs espaces par un seul
    # Exemple : "robot     piscine" -> "robot piscine"
    text = re.sub(r"\s+", " ", text)

    # Supprime les espaces au début et à la fin de la phrase
    return text.strip()

# Création d'une nouvelle colonne "text"
# On concatène le titre du produit (designation) et la description
# fillna("") permet de remplacer les valeurs NaN par une chaine vide
df["text"] = df["designation"].fillna("") + " " + df["description"].fillna("")

# Application de la fonction de nettoyage sur toute la colonne text
# Chaque ligne est nettoyée par la fonction clean_text
df["text_clean"] = df["text"].apply(clean_text)

C:\Users\user\AppData\Local\Temp\ipykernel_22132\1213809640.py:20: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text()


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Création du vectorizer TF-IDF
vectorizer = TfidfVectorizer(
    
    # Limite le vocabulaire aux 50 000 mots ou expressions les plus fréquents
    # Cela permet de réduire la dimension et d'améliorer les performances
    max_features=50000,
    
    # Utilisation des unigrammes et bigrammes
    # (1,2) signifie :
    # 1 mot : "piscine"
    # 2 mots : "robot piscine"
    ngram_range=(1,2),
    
    # Aucun mot stop n'est supprimé
    # On garde tous les mots car certains peuvent être utiles
    # pour la classification des produits
    stop_words=None
)

# Apprentissage du vocabulaire + transformation du texte en matrice TF-IDF
# Chaque ligne correspond à un produit
# Chaque colonne correspond à un mot ou un bigramme
X = vectorizer.fit_transform(df["text_clean"])

# Variable cible à prédire
# prdtypecode correspond à la catégorie du produit
y = df["prdtypecode"]


In [31]:
"""from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# Séparation des données en jeu d'entraînement et jeu de validation
# 80% pour entraîner le modèle et 20% pour l'évaluer
# stratify=y permet de garder la même proportion de classes dans les deux ensembles
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42  # pour rendre la séparation reproductible
)

# Création du modèle de régression logistique
# max_iter=1000 augmente le nombre d'itérations pour assurer la convergence
model = LogisticRegression(max_iter=1000)


# Entraînement du modèle sur les données d'entraînement
model.fit(X_train, y_train)


# Prédiction des catégories sur le jeu de validation
preds = model.predict(X_val)


# Calcul du score F1 pondéré
# Cette métrique est adaptée aux problèmes de classification multi-classes
# et correspond à la métrique utilisée dans le challenge Rakuten
print("F1 score :", f1_score(y_val, preds, average="weighted"))"""

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score,
    precision_score, recall_score,confusion_matrix
)
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

mlflow.set_tracking_uri("file:C:/Users/user/Rakuten-Challenge/mlruns")
mlflow.set_experiment("rakuten-classification")


# split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

with mlflow.start_run(run_name="logreg_baseline") as run:

    # modèle
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    # prédiction
    preds = model.predict(X_val)
    f1 = f1_score(y_val, preds, average="weighted")

    # logs
    mlflow.log_param("model", "logistic_regression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("max_features", 50000)
    mlflow.log_param("ngram_range", "(1,2)")

    mlflow.log_metric("f1_weighted", f1_score(y_val, preds, average="weighted"))
    mlflow.log_metric("f1_macro", f1_score(y_val, preds, average="macro"))
    mlflow.log_metric("f1_micro", f1_score(y_val, preds, average="micro"))

    mlflow.log_metric("accuracy", accuracy_score(y_val, preds))

    mlflow.log_metric("precision_weighted", precision_score(y_val, preds, average="weighted"))
    mlflow.log_metric("recall_weighted", recall_score(y_val, preds, average="weighted"))
    
    # prendre les 20 classes les plus fréquentes
    unique, counts = np.unique(y_val, return_counts=True)
    top_classes = unique[np.argsort(counts)[-20:]]

    cm = confusion_matrix(y_val, preds, labels=top_classes)

    plt.figure(figsize=(10,8))
    sns.heatmap(cm, cmap="Blues")
    plt.title("Confusion Matrix (Top 20 classes)")

    plt.savefig("confusion_matrix.png")

    mlflow.log_artifact("confusion_matrix.png")
    plt.close()
    
    eval_df = pd.DataFrame({
    "y_true": y_val,
    "y_pred": preds
    })

    mlflow.log_table(eval_df, "evaluation_table.json")

    # signature + input example
    input_example = X_train[:5]
    prediction_example = model.predict(input_example)
    signature = infer_signature(input_example, prediction_example)

    # log du modèle (UNE SEULE FOIS)
    mlflow.sklearn.log_model(
        model,
        "model",
        input_example=input_example,
        signature=signature
    )

    # run_id
    run_id = run.info.run_id

    # registry
    mlflow.register_model(
        f"runs:/{run_id}/model",
        "rakuten_model"
    )

    print("F1 score :", f1)


F1 score : 0.809959415827279


Registered model 'rakuten_model' already exists. Creating a new version of this model...
Created version '6' of model 'rakuten_model'.


In [32]:
import joblib

# Sauvegarde du modèle entraîné dans un fichier
# Cela permet de le réutiliser plus tard sans devoir le réentraîner
joblib.dump(model, "model.joblib")


# Sauvegarde du vectorizer TF-IDF
# Il est indispensable de sauvegarder aussi le vectorizer
# car il contient le vocabulaire appris pendant l'entraînement
joblib.dump(vectorizer, "vectorizer.joblib")

['vectorizer.joblib']

In [33]:
import os
print(os.getcwd())


c:\Users\user\Rakuten-Challenge\models\Models
